
# LangChain Prompts with Groq and Gradio

This notebook is a **Google Colab version** of the uploaded `langchain-prompts-main` project.

It has been transformed for:

- **Google Colab** instead of local VS Code only
- **Groq** instead of OpenAI
- **Gradio** instead of Streamlit
- No `uv`, no `.venv`, no `.env` required in Colab

## Original file mapping

| Original file | Colab/Groq/Gradio version |
|---|---|
| `chat_prompt_template.py` | Chat prompt template with Groq |
| `messages.py` | System, human, and AI messages with Groq |
| `prompt_template.py` | PromptTemplate with Groq |
| `temperature.py` | Temperature comparison with Groq |
| `message_placeholder.py` | MessagesPlaceholder with Groq |
| `prompt_generator.py` | Template generator |
| `prompt_ui.py` | Gradio research paper summarizer |
| `chatbot.py` | Gradio chatbot |



## 1. Install required packages

Run this cell once in Colab. Do not use `uv` or `venv` in Colab.


In [ ]:
%pip install -qU langchain langchain-core langchain-groq gradio


## 2. Configure Groq API key

Recommended: save your key in Colab Secrets as `GROQ_API_KEY`.

This cell also supports `Groq_API` if you already used that name.


In [ ]:
import os

try:
    from google.colab import userdata
    groq_api_key = userdata.get("Groq_API") or userdata.get("Groq_API")
except Exception:
    groq_api_key = None

if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key
    print("Groq API key configured successfully.")
else:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

Groq API key configured successfully.



## 3. Create the Groq model

We will reuse this model in the examples.


In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

print("Groq model is ready.")

Groq model is ready.



# Example 1 — Chat Prompt Template

This replaces the original `chat_prompt_template.py`.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful {domain} expert."),
    ("human", "Explain in simple terms: what is {topic}?")
])

prompt = chat_template.invoke({
    "domain": "Medical",
    "topic": "Polio"
})

result = model.invoke(prompt)
print(result.content)

Polio, also known as Poliomyelitis, is a serious and highly infectious disease caused by a virus. It affects the nervous system, which controls the muscles and nerves in our body.

Here's how it works:

1. **The virus enters the body**: Polio virus is usually spread through contaminated food, water, or direct contact with an infected person.
2. **The virus attacks the nervous system**: Once inside the body, the virus attacks the nerve cells, which can lead to muscle weakness, paralysis, and even death.
3. **The virus can cause different symptoms**: Some people may not show any symptoms at all, while others may experience:
	* Mild symptoms like fever, headache, and muscle pain
	* Severe symptoms like paralysis, muscle weakness, and respiratory problems
	* In rare cases, the virus can cause polio meningitis, which is an inflammation of the lining around the brain and spinal cord

**Types of Polio:**

1. **Paralytic Polio**: This is the most severe form of the disease, where the virus cau


# Example 2 — Messages

This replaces the original `messages.py`.

It uses:

- `SystemMessage` for model role
- `HumanMessage` for user input
- `AIMessage` to store model response


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Tell me about LangChain in simple words.")
]

result = model.invoke(messages)
messages.append(AIMessage(content=result.content))

print(result.content)
print("Conversation Messages:")
print(messages)

LangChain is an open-source framework for building large language models (LLMs) and their applications. It's designed to make it easier for developers to work with LLMs and create more complex AI models.

Think of LangChain like a toolbox that helps you:

1. **Connect** multiple LLMs together to create more powerful models.
2. **Integrate** LLMs with other AI technologies, like databases or APIs.
3. **Build** custom applications using LLMs, like chatbots or virtual assistants.

LangChain provides a set of tools and libraries that make it easier to work with LLMs, allowing developers to focus on building innovative applications rather than dealing with the underlying complexities of LLMs.

In simple terms, LangChain is like a bridge that connects LLMs to other technologies, making it easier to build more advanced AI models and applications.
Conversation Messages:
[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tel


# Example 3 — Prompt Template

This replaces the original `prompt_template.py`.


In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template="Greet this person in 5 languages. The name of the person is {name}.",
    input_variables=["name"]
)

prompt = template.invoke({"name": "Sajid"})
result = model.invoke(prompt)

print(result.content)

Here are greetings in 5 different languages for Sajid:

1. **English**: Hello Sajid, how are you?
2. **Spanish**: Hola Sajid, ¿cómo estás?
3. **French**: Bonjour Sajid, comment vas-tu?
4. **German**: Hallo Sajid, wie geht es dir?
5. **Arabic**: مرحبا ساجد (Marhaba Sajid), كيف حالك؟



# Example 4 — Temperature

This replaces the original `temperature.py`.

Temperature controls creativity:

- `0.0` = more consistent
- `0.7` = balanced
- `1.5` = more creative


In [ ]:

prompt = "Write a 5-line poem on cricket."

for temp in [0.0, 0.7, 1.5]:
    temp_model = ChatGroq(model="llama-3.1-8b-instant", temperature=temp)
    result = temp_model.invoke(prompt)
    print("=" * 60)
    print(f"Temperature: {temp}")
    print(result.content)


Temperature: 0.0
The cricket field, a sight to see,
Players in whites, a symphony.
The ball flies through, a swift delight,
The batsman's skill, a wondrous sight,
A game of skill, in the warm sun's light.
Temperature: 0.7
The cricket field, a sight to see,
Players in whites, a symphony.
The ball flies high, the wickets fall,
A game of skill, played by all,
Summer's joy, in every call.
Temperature: 1.5
The ball is tossed, the match is near,
The cricketers face, their skills clear.
With bat and ball, they give their best,
The score unfolds, the thrill takes rest,
A game of skill, a game of fun to hear.



# Example 5 — Messages Placeholder

This replaces the original `message_placeholder.py`.

`MessagesPlaceholder` is used when we want to insert previous conversation history into a prompt.


In [ ]:
from langchain_core.prompts import MessagesPlaceholder

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support agent."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}")
])

chat_history = [
    HumanMessage(content="I want to request a refund for my order #12345."),
    AIMessage(content="Your refund request for order #12345 has been initiated. It will be processed in 3-5 business days.")
]

prompt = chat_template.invoke({
    "chat_history": chat_history,
    "query": "Where is my refund?"
})

result = model.invoke(prompt)
print(result.content)

I've checked on the status of your refund for order #12345. It appears that the refund was processed successfully, but it may take a few more days to reflect in your account due to bank processing times.

To confirm, I've checked the refund details:

- Order Number: #12345
- Refund Amount: $X.XX
- Refund Method: Original payment method (credit/debit card)
- Refund Status: Completed

If you haven't received the refund yet, I recommend checking with your bank or credit card issuer to confirm their processing times. If you have any further questions or concerns, please let me know.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Prompt template
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support agent."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{query}")
])

# Store chat history
chat_history = []

# Function to chat while keeping only the last 3 messages
def chat(query):
    global chat_history

    # Build prompt with last 3 messages
    prompt = chat_template.invoke({
        "chat_history": chat_history[-3:],
        "query": query
    })

    # Get model response
    response = model.invoke(prompt)

    # Update history
    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=response.content)
    ])

    # Keep only the latest 3 messages
    chat_history = chat_history[-3:]

    return response.content

# Example usage
print(chat("I want to request a refund for my order #12345."))
print("==========================================================")
print(chat("Where is my refund?"))
print("==========================================================")
print(chat("How long will it take?"))

I'd be happy to help you with your refund request. Can you please provide me with a bit more information about your order? 

Here are a few details I'll need to assist you:

1. What is the reason for your refund request? (e.g. incorrect item, damaged item, not as described, etc.)
2. When did you place the order (date)?
3. What is the approximate date you received the order?
4. Have you already contacted us about this issue previously?

Once I have this information, I'll do my best to assist you with your refund request.
I've checked on the status of your refund request for order #12345. 

To confirm, I've located your order in our system, and I see that we processed a refund for you on [date]. The refund amount was [amount] and it was issued back to the original payment method.

However, I also noticed that there might be a delay in the refund processing time due to [possible reason, e.g. bank processing time, payment method restrictions, etc.]. 

If you haven't received the refund yet


# Example 6 — Prompt Generator

This replaces the original `prompt_generator.py`.

It creates a reusable research-paper summary template.


In [ ]:
from langchain_core.prompts import PromptTemplate

research_template = PromptTemplate(
    template="""
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}

Requirements:
1. Explain the main idea clearly.
2. Mention the problem the paper is trying to solve.
3. Mention the method or model used in the paper.
4. Mention important mathematical ideas only if they are relevant.
5. Use simple analogies when helpful.
6. If certain information is not available, respond with: "Insufficient information available" instead of guessing.

Ensure the summary is clear, accurate, and aligned with the selected style and length.
""",
    input_variables=["paper_input", "style_input", "length_input"],
    validate_template=True,
)

print("Research prompt template created successfully.")


Research prompt template created successfully.



# Example 7 — Research Tool with Gradio

This replaces the original Streamlit file `prompt_ui.py`.

In Colab, Gradio is easier because it creates a shareable web link.


In [ ]:
import gradio as gr

research_chain = research_template | model

papers = [
    "Attention Is All You Need",
    "BERT: Pre-training of Deep Bidirectional Transformers",
    "GPT-3: Language Models are Few-Shot Learners",
    "Diffusion Models Beat GANs on Image Synthesis",
    "Integration of Federated Learning and Blockchain in Health Care: Tutorial on Medical Data, Architectures, Privacy, Security, and Regulatory Compliance",
    "Sahi-al-Bukhari"
]

styles = ["Beginner-Friendly", "Technical", "Code-Oriented", "Mathematical"]
lengths = ["Short (1-2 paragraphs)", "Medium (3-5 paragraphs)", "Long (detailed explanation)"]


def summarize_paper(paper_input, style_input, length_input):
    result = research_chain.invoke({
        "paper_input": paper_input,
        "style_input": style_input,
        "length_input": length_input,
    })
    return result.content

research_demo = gr.Interface(
    fn=summarize_paper,
    inputs=[
        gr.Dropdown(papers, label="Select Research Paper Name"),
        gr.Dropdown(styles, label="Select Explanation Style"),
        gr.Dropdown(lengths, label="Select Explanation Length"),
    ],
    outputs=gr.Markdown(label="Summary"),
    title="Research Paper Summarizer using Groq + LangChain",
    description="A Colab-friendly Gradio version of the original Streamlit research tool.",
)

research_demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b2d8273377595bb384.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b2d8273377595bb384.gradio.live



# Example 8 — Chatbot with Gradio

This replaces the original terminal-based `chatbot.py`.

It keeps conversation history during the Gradio session.


In [ ]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

def chat(message, history):
    messages = [
        SystemMessage(
            content="You are a helpful AI assistant. Answer clearly and practically."
        )
    ]

    # Gradio message format:
    # [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")

        if role == "user":
            messages.append(HumanMessage(content=content))

        elif role == "assistant":
            messages.append(AIMessage(content=content))

    # Add current user message
    messages.append(HumanMessage(content=message))

    response = chat_model.invoke(messages)

    return response.content


demo = gr.ChatInterface(
    fn=chat,
    title="Groq Chatbot using LangChain + Gradio",
    description="Full conversation history is sent to the model."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9a01c3248535536813.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 


# Student Practice

1. Change the domain and topic in the chat prompt template.
2. Create your own system message for a teacher, doctor, business manager, or researcher.
3. Compare temperature outputs for the same prompt.
4. Modify the research paper summarizer for your own field.
5. Customize the chatbot system message for a specific use case.

## Important

Do not share your API key in screenshots, notebooks, or class groups.


In [ ]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)

# -----------------------------
# LLM
# -----------------------------
chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

SYSTEM_PROMPT = """
You are an expert AI assistant.

Answer professionally.
Use markdown.
Use bullet points whenever useful.
"""

# -----------------------------
# Chat Function
# -----------------------------
def respond(message, history):

    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    for human, assistant in history:
        messages.append(HumanMessage(content=human))
        messages.append(AIMessage(content=assistant))

    messages.append(HumanMessage(content=message))

    response = chat_model.invoke(messages)

    history.append((message, response.content))

    return history, ""


# -----------------------------
# Theme
# -----------------------------
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="gray",
)


css = """
.gradio-container{
    max-width:1200px;
    margin:auto;
}

footer{
    display:none;
}

#title{
    text-align:center;
    font-size:34px;
    font-weight:700;
    margin-bottom:10px;
}

#subtitle{
    text-align:center;
    color:gray;
    margin-bottom:25px;
}

.message{
    border-radius:14px !important;
}
"""


# -----------------------------
# UI
# -----------------------------
with gr.Blocks(theme=theme, css=css) as demo:

    gr.Markdown(
        "# 🤖 AI Assistant",
        elem_id="title"
    )

    gr.Markdown(
        "Powered by **Groq + LangChain**",
        elem_id="subtitle"
    )

    chatbot = gr.Chatbot(
        height=650,
        bubble_full_width=False
    )

    with gr.Row():

        textbox = gr.Textbox(
            placeholder="Ask me anything...",
            scale=8
        )

        send = gr.Button(
            "Send",
            variant="primary",
            scale=1
        )

    clear = gr.Button("🗑 Clear Chat")

    examples = gr.Examples(
        examples=[
            "Explain RAG",
            "What is Agentic AI?",
            "Write FastAPI CRUD API",
            "Explain Vector Databases",
            "Build an AI chatbot architecture"
        ],
        inputs=textbox
    )

    history = gr.State([])

    send.click(
        respond,
        inputs=[textbox, history],
        outputs=[chatbot, textbox]
    )

    textbox.submit(
        respond,
        inputs=[textbox, history],
        outputs=[chatbot, textbox]
    )

    clear.click(
        lambda: ([], []),
        outputs=[chatbot, history]
    )

demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_31152/2461852605.py:86: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css) as demo:


TypeError: Chatbot.__init__() got an unexpected keyword argument 'bubble_full_width'

In [ ]:
import gradio as gr

print(gr.__version__)
print(gr.__file__)

6.20.0
/usr/local/lib/python3.12/dist-packages/gradio/__init__.py
